# BUS-BRA: shared 20-image Kaggle smoke test

This standalone notebook implements **Pilots 1 and 2**, using the same 20 original
BUS-BRA images (10 benign, 10 malignant; 20 distinct source `Case` IDs). It does
**not** run the 400-image final benchmark. Check feasibility and output behavior
before choosing final settings. GPU execution must still be validated on Kaggle.

**Default matrix, executed sequentially:** Qwen2-VL and MedVLM-R1 in FP16;
Gemma 3 and MedGemma each in FP32 and NF4 4-bit weights / FP32 compute.
This is six jobs and up to **120 responses**. One model is loaded at a time.
The initial **32-token limit is preserved**, even if a model fails to finish.
Failed loads/OOMs and invalid/truncated answers are evidence to review, not reasons
to silently switch precision or extend generation.

### Kaggle setup
1. Create a notebook and use **File → Import Notebook** to upload this file.
2. Attach the original BUS-BRA Kaggle dataset with **Add Input**. It must contain
   `bus_data.csv` and original `Images/`, or the original BUS-BRA ZIP. Dataset
   attachment slugs can vary: discovery does not assume a specific Kaggle slug.
3. Choose **GPU T4 x2**, if available, and enable **Internet** in notebook settings.
   Two GPUs give the non-quantized Gemma FP32 jobs a chance to fit. One GPU is
   allowed: an unsupported/OOM FP32 job is recorded and the planned 4-bit job is tried.
4. Accept access terms for [Gemma 3](https://huggingface.co/google/gemma-3-4b-it)
   and [MedGemma](https://huggingface.co/google/medgemma-4b-it) using your own account.
   In **Add-ons → Secrets**, add a read-access Hugging Face token named `HF_TOKEN`
   and enable it for this notebook. Do not put credentials in cells.
5. Use a fresh Python session. Edit the configuration only as described, then **Run All**.
   The source audit may take a few minutes; models must download on their first load.
6. Save the notebook version with outputs and download
   `/kaggle/working/bus_bra_smoke_results.zip` from the final cell or Output files.

If you already have a pilot manifest/result folder from an earlier run, attach it
and configure `FROZEN_PILOT_CSV` / `RESUME_FROM` before running. Reuse its pinned
model revisions. Review raw output, failure logs, manifest and contact sheet
before final evaluation. This notebook never chooses a scientific winner.

## 1. Configuration

Keep the six default jobs for the complete pilot. To split work across sessions,
set `JOBS` to a subset, e.g. `[('qwen2_vl', 'fp16'), ('medvlm_r1', 'fp16')]`.
Each later session should attach the previous output folder and set `RESUME_FROM`.
Alternatively attach its `pilot_manifest.csv` as `FROZEN_PILOT_CSV` and explicitly
reuse `model_revisions.json` values when doing independent runs.

`DATA_ROOT=None` discovers the dataset automatically. If more than one copy is
attached, set it to the directory containing the desired `bus_data.csv`.
Only original images are used. The local repository's `Datasets/` is untouched.

The source file fingerprints are checked against the locally inspected original.
A reformatted or modified Kaggle mirror stops with a clear error. Send that error
for review or attach the unmodified original archive; do not bypass the check.
Downloaded model caches are removed after the last job for each model to save disk.
The results bundle contains no model weights or full dataset.

In [ ]:
from pathlib import Path

# Run the complete pilot matrix, or select a subset to fit a Kaggle session.
JOBS = [
    ("qwen2_vl", "fp16"),
    ("medvlm_r1", "fp16"),
    ("gemma3", "fp32"),
    ("gemma3", "int4_fp32"),
    ("medgemma", "fp32"),
    ("medgemma", "int4_fp32"),
]
DATA_ROOT = None          # Auto-discover bus_data.csv under /kaggle/input.
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/bus_bra_smoke")
EXTRACT_ROOT = Path("/kaggle/working/bus_bra_extracted")
MODEL_CACHE_ROOT = Path("/kaggle/working/bus_bra_model_cache")
CLEAR_MODEL_CACHE_AFTER_LAST_JOB = True  # Remove this notebook's downloaded weights to save disk.
FROZEN_PILOT_CSV = None   # Optional attached pilot_manifest.csv from an earlier run.
RESUME_FROM = None        # Optional attached bus_bra_smoke directory, not its ZIP.
INSTALL_DEPENDENCIES = True

SEED = 42
PILOT_SIZE = 20
MAX_NEW_TOKENS = 32       # Agreed initial limit; never auto-increased.
NEAR_DUPLICATE_DISTANCE = 4  # Conservative dHash review candidates, not a diagnosis.
GPU_MEMORY_FRACTION = 0.80  # Reserve space for image activations / KV cache.
PILOT_CODE_VERSION = "bus-bra-smoke-v1"
PARSER_VERSION = "strict-final-v1"

MODEL_IDS = {
    "qwen2_vl": "Qwen/Qwen2-VL-2B-Instruct",
    "medvlm_r1": "JZPeterPan/MedVLM-R1",
    "gemma3": "google/gemma-3-4b-it",
    "medgemma": "google/medgemma-4b-it",
}
# None resolves once to a Hub commit, saved in model_revisions.json.
# When resuming on a new session, attach the previous output directory.
MODEL_REVISIONS = {key: None for key in MODEL_IDS}
PACKAGE_PINS = {
    "transformers": "4.57.1", "accelerate": "1.11.0",
    "bitsandbytes": "0.48.1", "huggingface-hub": "0.36.0",
    "tokenizers": "0.22.1", "safetensors": "0.6.2",
    "sentencepiece": "0.2.1", "Pillow": "11.3.0",
}
# Keep Kaggle's matched torch/torchvision CUDA runtime; record its exact versions.
BUS_BRA_PROMPT = """You are a medical assistant. Examine this breast ultrasound image.
Classify the lesion as exactly ONE of the following:
- benign: non-cancerous lesion
- malignant: cancerous lesion

Respond with ONLY the label: benign or malignant."""

assert PILOT_SIZE == 20 and MAX_NEW_TOKENS == 32
assert JOBS and len(JOBS) == len(set(JOBS))
for key, precision in JOBS:
    assert key in MODEL_IDS
    assert precision in ({"fp16"} if key in {"qwen2_vl", "medvlm_r1"}
                         else {"fp32", "int4_fp32"})


# Embedded build provenance; actual pilot behavior is identified by the code hash.
IMPLEMENTATION_SHA256 = '9fcb31923eafb2f92497af864ca559d5284c7ca82c4d2fea6b53f4bf7724c811'
CODE_GIT_COMMIT = '2af45c377f84658bfa5666c932b4b56625cc4fcc'  # Parent commit at notebook build time.


## 2. Install the pilot dependency versions

These are explicit **candidate pilot versions**, not a claim of tested GPU
compatibility. Kaggle's existing torch/torchvision installation is retained and
constrained during installation, then recorded with CUDA/hardware details.
Run installation before importing ML packages. If a changed version was already
imported, restart the session and Run All. Dependency/network errors here must be
resolved before the GPU pilot can begin.

In [ ]:
import importlib.metadata as im
import subprocess
import sys

if INSTALL_DEPENDENCIES:
    # Run this cell before importing transformers/torch/PIL in a fresh session.
    changed_loaded = []
    for package, module in {"transformers": "transformers", "Pillow": "PIL",
                            "tokenizers": "tokenizers"}.items():
        try:
            if module in sys.modules and im.version(package) != PACKAGE_PINS[package]:
                changed_loaded.append(package)
        except im.PackageNotFoundError:
            pass
    if changed_loaded:
        raise RuntimeError("Restart the Kaggle session before installing: " + ", ".join(changed_loaded))
    # Constrain torch packages so pip cannot replace Kaggle's CUDA installation.
    constraints = Path("/tmp/bus_bra_runtime_constraints.txt")
    constraints.write_text("\n".join(
        f"{name}=={im.version(name)}" for name in ("torch", "torchvision")
    ) + "\n")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "--disable-pip-version-check", "-c", str(constraints),
                    *[f"{name}=={version}" for name, version in PACKAGE_PINS.items()]], check=True)
    importlib = __import__("importlib")
    importlib.invalidate_caches()
for name, expected in PACKAGE_PINS.items():
    if im.version(name) != expected:
        raise RuntimeError(f"Dependency mismatch: {name}; expected {expected}.")
print("Pilot dependencies installed. Kaggle torch/torchvision were preserved.")



## 3. Data, persistence, and audit helpers

Read the original metadata, verify image hashes/readability/dimensions, use `Case`
for source grouping, and flag inconsistent metadata or cross-case duplicate
candidates. Near-duplicate candidates use 64-bit dHash distance ≤ 4 and require
later review; this threshold is not proof of semantic duplication. Flagged cases
are retained in audit records and excluded from this preliminary engineering pilot.
No pathology is re-labeled, and no model predictions influence sample selection.

The local audit found 1,875 images and 1,064 distinct source cases. Patient identity
and pathology cannot be independently established from files alone. Final sampling
still requires review of the audit and the pilot contact sheet.

In [ ]:
import csv
import gc
import hashlib
import io
import json
import os
import platform
import random
import re
import shutil
import statistics
import time
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from PIL import Image, ImageDraw, ImageStat

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha_bytes(value):
    return hashlib.sha256(value).hexdigest()

def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def object_hash(value):
    return sha_bytes(json.dumps(value, sort_keys=True, separators=(",", ":"),
                               ensure_ascii=False).encode())

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n")
    os.replace(temp, path)

def write_csv(path, rows, fields=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = fields or (list(rows[0]) if rows else ["status"])
    temp = path.with_suffix(path.suffix + ".tmp")
    with temp.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="raise")
        writer.writeheader()
        writer.writerows(rows)
    os.replace(temp, path)

def read_csv(path):
    with Path(path).open(newline="", encoding="utf-8-sig") as stream:
        return list(csv.DictReader(stream))

def append_jsonl(path, row):
    with Path(path).open("a") as stream:
        stream.write(json.dumps(row, sort_keys=True, allow_nan=False) + "\n")
        stream.flush()
        os.fsync(stream.fileno())

def read_jsonl(path, repair_tail=False):
    path = Path(path)
    if not path.exists():
        return []
    lines = path.read_bytes().splitlines(keepends=True)
    rows, offset = [], 0
    for i, line in enumerate(lines):
        try:
            row = json.loads(line)
        except (ValueError, UnicodeDecodeError):
            if not (repair_tail and i == len(lines) - 1 and not line.endswith(b"\n")):
                raise ValueError(f"Corrupt complete record in {path.name}, line {i + 1}")
            # Preserve the broken tail for inspection; retain all complete records.
            path.with_suffix(".partial_tail").write_bytes(line)
            with path.open("r+b") as stream:
                stream.truncate(offset)
            break
        if not isinstance(row, dict):
            raise ValueError("JSONL records must be objects")
        rows.append(row)
        offset += len(line)
    if repair_tail and rows and path.stat().st_size and not path.read_bytes().endswith(b"\n"):
        with path.open("ab") as stream:
            stream.write(b"\n")
    return rows

def restore_outputs(source, destination):
    source, destination = Path(source), Path(destination)
    if not source.is_dir() or source.resolve() == destination.resolve():
        raise ValueError("RESUME_FROM must name a different, existing output directory")
    destination.mkdir(parents=True, exist_ok=True)
    for item in source.rglob("*"):
        if item.is_symlink():
            raise ValueError("Symlinks are not supported in resume artifacts")
        if not item.is_file():
            continue
        target = destination / item.relative_to(source)
        if target.exists():
            if sha_file(target) != sha_file(item):
                raise ValueError(f"Resume conflict: {target.name}; use a fresh OUTPUT_ROOT")
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, target)

def discover_dataset(data_root, input_root, extract_root):
    if data_root is not None:
        root = Path(data_root)
        candidates = list(root.rglob("bus_data.csv"))
    else:
        candidates = list(Path(input_root).rglob("bus_data.csv"))
    if len(candidates) == 1:
        return candidates[0].parent
    if len(candidates) > 1:
        raise ValueError("Multiple BUS-BRA metadata files found; set DATA_ROOT explicitly")
    if data_root is not None:
        raise FileNotFoundError("DATA_ROOT must contain the original bus_data.csv")
    archives = []
    for path in Path(input_root).rglob("*.zip"):
        if "bus" not in path.name.lower():
            continue
        with zipfile.ZipFile(path) as archive:
            if any(Path(n).name == "bus_data.csv" for n in archive.namelist()):
                archives.append(path)
    if len(archives) != 1:
        raise FileNotFoundError("Attach a BUS-BRA dataset with bus_data.csv and original Images, or its ZIP. Set DATA_ROOT if ambiguous.")
    extract_root = Path(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            parts = Path(member.filename).parts
            # Only original images, metadata and license; no masks or CV files needed.
            wanted = "images" in [p.lower() for p in parts] or Path(member.filename).name in {"bus_data.csv", "LICENSE.txt"}
            if member.is_dir() or not wanted:
                continue
            target = (extract_root / member.filename).resolve()
            if not target.is_relative_to(extract_root.resolve()):
                raise ValueError("Unsafe ZIP member")
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as src, target.open("wb") as dst:
                shutil.copyfileobj(src, dst)
    candidates = list(extract_root.rglob("bus_data.csv"))
    if len(candidates) != 1:
        raise ValueError("Ambiguous extracted dataset; set DATA_ROOT explicitly")
    return candidates[0].parent

def dhash(image):
    pixels = list(image.convert("L").resize((9, 8), Image.Resampling.LANCZOS).getdata())
    result = 0
    for y in range(8):
        for x in range(8):
            result = (result << 1) | int(pixels[y * 9 + x] > pixels[y * 9 + x + 1])
    return result

MANIFEST_FIELDS = ["dataset", "sample_id", "image_id", "image_path", "group_id", "split",
                   "task_type", "question", "options_json", "reference_answer", "category", "image_sha256"]

def audit_dataset(root, output, expected_metadata_sha):
    metadata_path = root / "bus_data.csv"
    metadata_sha = sha_file(metadata_path)
    if metadata_sha != expected_metadata_sha:
        raise ValueError("Kaggle bus_data.csv differs from the locally inspected original. Use an unmodified BUS-BRA copy; send this error for review rather than guessing a label mapping.")
    rows = read_csv(metadata_path)
    required = {"ID", "Case", "Pathology", "Histology", "Width", "Height"}
    if not rows or not required <= rows[0].keys():
        raise ValueError("Missing original BUS-BRA metadata columns")
    if len(rows) != 1875 or len({r["Case"] for r in rows}) != 1064:
        raise ValueError("Unexpected original BUS-BRA metadata counts")
    dirs = [p for p in root.iterdir() if p.is_dir() and p.name.lower() == "images"]
    if len(dirs) != 1:
        raise ValueError("Expected one original Images directory beside bus_data.csv")
    image_index = defaultdict(list)
    for path in dirs[0].rglob("*"):
        if path.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"}:
            image_index[path.stem].append(path)
    ids = Counter(r["ID"] for r in rows)
    case_labels = defaultdict(set)
    histology_labels = defaultdict(set)
    for r in rows:
        case_labels[r["Case"]].add(r["Pathology"].strip().lower())
        histology_labels[r["Histology"].strip().lower()].add(r["Pathology"].strip().lower())
    audited = []
    for r in sorted(rows, key=lambda x: x["ID"]):
        flags = []
        label = r["Pathology"].strip().lower()
        if label not in {"benign", "malignant"}: flags.append("unknown_label")
        if not r["Case"].strip(): flags.append("missing_case")
        if ids[r["ID"]] != 1: flags.append("duplicate_metadata_id")
        if len(case_labels[r["Case"]]) != 1: flags.append("mixed_case_labels_review")
        if not r["Histology"].strip(): flags.append("missing_histology_review")
        # This flags ambiguity for review; it does not adjudicate a medical diagnosis.
        if len(histology_labels[r["Histology"].strip().lower()]) > 1:
            flags.append("histology_label_mapping_review")
        record = {"sample_id": r["ID"], "group_id": r["Case"], "reference_answer": label,
                  "image_path": "", "image_sha256": "", "pixel_sha256": "", "dhash": "",
                  "flags": flags}
        paths = image_index[r["ID"]]
        if len(paths) != 1:
            flags.append("missing_or_ambiguous_image")
        else:
            path = paths[0]
            record.update(image_path=path.relative_to(root).as_posix(), image_sha256=sha_file(path))
            try:
                with Image.open(path) as im:
                    im.load()
                    if im.size != (int(r["Width"]), int(r["Height"])):
                        flags.append("metadata_dimensions_mismatch")
                    if min(im.size) < 32 or ImageStat.Stat(im.convert("L")).stddev[0] < 1:
                        flags.append("severely_degraded_candidate")
                    rgb = im.convert("RGB")
                    record["pixel_sha256"] = sha_bytes(str(rgb.size).encode() + rgb.tobytes())
                    record["dhash"] = f"{dhash(rgb):016x}"
            except Exception as exc:
                flags.append("unreadable_image:" + type(exc).__name__)
        audited.append(record)
    duplicates = []
    for i, a in enumerate(audited):
        if not a["dhash"]: continue
        for b in audited[i + 1:]:
            if not b["dhash"]: continue
            exact = a["pixel_sha256"] == b["pixel_sha256"]
            distance = (int(a["dhash"], 16) ^ int(b["dhash"], 16)).bit_count()
            if exact or distance <= NEAR_DUPLICATE_DISTANCE:
                cross = a["group_id"] != b["group_id"]
                duplicates.append({"sample_a": a["sample_id"], "sample_b": b["sample_id"],
                                   "exact_pixels": exact, "dhash_distance": distance,
                                   "cross_case": cross})
                if cross:
                    for r in (a, b):
                        r["flags"].append("cross_case_duplicate_candidate")
    # Be conservative for this engineering pilot: reserve flagged cases for later review.
    flagged_cases = {r["group_id"] for r in audited if r["flags"]}
    for r in audited:
        if r["group_id"] in flagged_cases and not r["flags"]:
            r["flags"].append("another_image_in_case_flagged")
        r["flags"] = ";".join(sorted(set(r["flags"])))
    fingerprint = object_hash({"metadata_sha256": metadata_sha,
                               "files": [(r["sample_id"], r["image_sha256"]) for r in audited]})
    eligible = [r for r in audited if not r["flags"]]
    summary = {"metadata_sha256": metadata_sha, "source_fingerprint": fingerprint,
               "total_images": len(audited), "total_cases": len(case_labels),
               "eligible_pilot_images": len(eligible), "flagged_cases": len(flagged_cases),
               "eligible_cases_by_class": {label: len({r["group_id"] for r in eligible if r["reference_answer"] == label})
                                           for label in ("benign", "malignant")},
               "near_duplicate_method": "64-bit dHash", "near_duplicate_distance": NEAR_DUPLICATE_DISTANCE,
               "duplicate_candidates": len(duplicates),
               "limitations": ["Case is the source dataset's grouping field, not independently verified patient identity.",
                               "Automated screening is preliminary; review contact sheet and flagged cases before final sampling.",
                               "Histology consistency is a metadata check, not independent pathology verification."]}
    output.mkdir(parents=True, exist_ok=True)
    write_csv(output / "audit_records.csv", audited)
    write_csv(output / "duplicate_candidates.csv", duplicates,
              ["sample_a", "sample_b", "exact_pixels", "dhash_distance", "cross_case"])
    write_json(output / "audit_summary.json", summary)
    return audited, summary

def make_pilot(audited, frozen_path=None):
    eligible = {r["sample_id"]: r for r in audited if not r["flags"]}
    if frozen_path is not None:
        pilot = read_csv(frozen_path)
    else:
        selected = []
        rng = random.Random(SEED)
        for label in ("benign", "malignant"):
            by_case = defaultdict(list)
            for r in eligible.values():
                if r["reference_answer"] == label:
                    by_case[r["group_id"]].append(r)
            if len(by_case) < 10:
                raise ValueError(f"Fewer than 10 eligible cases for {label}")
            for case in rng.sample(sorted(by_case), 10):
                selected.append(rng.choice(sorted(by_case[case], key=lambda r: r["sample_id"])))
        pilot = [{"dataset": "bus_bra", "sample_id": r["sample_id"], "image_id": r["sample_id"],
                  "image_path": r["image_path"], "group_id": r["group_id"], "split": "pilot",
                  "task_type": "classification", "question": BUS_BRA_PROMPT, "options_json": "{}",
                  "reference_answer": r["reference_answer"], "category": "", "image_sha256": r["image_sha256"]}
                 for r in sorted(selected, key=lambda r: r["sample_id"])]
    if len(pilot) != 20 or len({r["sample_id"] for r in pilot}) != 20 or len({r["group_id"] for r in pilot}) != 20:
        raise ValueError("Pilot must contain exactly 20 unique images from 20 distinct cases")
    if Counter(r["reference_answer"] for r in pilot) != {"benign": 10, "malignant": 10}:
        raise ValueError("Pilot must contain 10 images per class")
    for r in pilot:
        if set(r) != set(MANIFEST_FIELDS): raise ValueError("Unexpected manifest schema")
        current = eligible.get(r["sample_id"])
        if current is None: raise ValueError("Frozen pilot contains an ineligible/missing record")
        for key in ("group_id", "reference_answer", "image_path", "image_sha256"):
            if r[key] != current[key]: raise ValueError(f"Frozen pilot mismatch: {key}")
        if r["split"] != "pilot" or r["dataset"] != "bus_bra" or r["question"] != BUS_BRA_PROMPT:
            raise ValueError("Not the agreed BUS-BRA pilot manifest")
    return pilot

def contact_sheet(root, pilot, destination):
    sheet = Image.new("RGB", (1000, 5 * 240), "white")
    draw = ImageDraw.Draw(sheet)
    for i, row in enumerate(pilot):
        x, y = (i % 4) * 250, (i // 4) * 240
        with Image.open(root / row["image_path"]) as im:
            thumbnail = im.convert("RGB")
            thumbnail.thumbnail((240, 190))
            sheet.paste(thumbnail, (x + 5, y + 5))
        draw.text((x + 5, y + 199), row["sample_id"], fill="black")
        draw.text((x + 5, y + 216), row["reference_answer"], fill="black")
    sheet.save(destination)



## 4. Freeze the shared pilot and inspect its contact sheet

The deterministic sampler first selects 10 eligible cases per class, then one
image per case, with seed 42. All jobs use the resulting single manifest. Its
hash and the reserved case list are exported. No final-test set is created here.
Reserve all images from these cases when building the future test set.

The contact sheet is for screening only; inference always uses original images.
If an image appears unsuitable, stop and document the issue before changing the
manifest. Do not replace cases after inspecting model performance.

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if RESUME_FROM is not None:
    restore_outputs(RESUME_FROM, OUTPUT_ROOT)
SOURCE_ROOT = discover_dataset(DATA_ROOT, INPUT_ROOT, EXTRACT_ROOT)
if (SOURCE_ROOT / "LICENSE.txt").exists():
    shutil.copy2(SOURCE_ROOT / "LICENSE.txt", OUTPUT_ROOT / "BUS_BRA_LICENSE.txt")
AUDITED, AUDIT = audit_dataset(
    SOURCE_ROOT, OUTPUT_ROOT / "audit",
    "f218d0f10b0aeeb14ff4b2cfed8631b2f4a7cf608e564548b35bddd55f3aafb3",
)
if AUDIT["source_fingerprint"] != "e1fadc2a04595e1e17c9e8e7e72def7ffa954d52b4de5c386398623286848b49":
    raise ValueError("Kaggle image files differ from the locally audited source; use the original BUS-BRA copy or review the differences before sampling")
manifest_path = OUTPUT_ROOT / "pilot_manifest.csv"
existing_manifest = manifest_path if manifest_path.exists() else FROZEN_PILOT_CSV
PILOT = make_pilot(AUDITED, existing_manifest)
if FROZEN_PILOT_CSV is not None and manifest_path.exists():
    supplied = make_pilot(AUDITED, FROZEN_PILOT_CSV)
    if supplied != PILOT:
        raise ValueError("Attached pilot differs from existing outputs; use a fresh OUTPUT_ROOT")
if not manifest_path.exists():
    write_csv(manifest_path, PILOT, MANIFEST_FIELDS)
MANIFEST_SHA = sha_file(manifest_path)
pilot_groups = {r["group_id"] for r in PILOT}
write_csv(OUTPUT_ROOT / "reserved_pilot_cases.csv", [
    {"sample_id": r["sample_id"], "group_id": r["group_id"], "image_sha256": r["image_sha256"]}
    for r in AUDITED if r["group_id"] in pilot_groups
])
write_json(OUTPUT_ROOT / "manifest_metadata.json", {
    "purpose": "engineering_pilot_only", "seed": SEED, "images": 20, "distinct_cases": 20,
    "class_counts": dict(Counter(r["reference_answer"] for r in PILOT)),
    "manifest_sha256": MANIFEST_SHA, "source_fingerprint": AUDIT["source_fingerprint"],
    "metadata_sha256": AUDIT["metadata_sha256"],
    "grouping_field": "Case", "manual_screening": "pending team review",
    "final_test_manifest": "not_created",
})
contact_sheet(SOURCE_ROOT, PILOT, OUTPUT_ROOT / "pilot_contact_sheet.jpg")
print("Data root:", SOURCE_ROOT)
print("Eligible cases by class:", AUDIT["eligible_cases_by_class"])
print("Frozen pilot:", len(PILOT), "images from", len(pilot_groups), "distinct cases")
print("Manifest SHA256:", MANIFEST_SHA)
from IPython.display import display, Image as DisplayImage
display(DisplayImage(filename=str(OUTPUT_ROOT / "pilot_contact_sheet.jpg"), width=900))



## 5. Strict final-answer parser and synthetic checks

Accept an exact case-insensitive label, `Final answer: <label>`, or exactly one
complete `<answer>` element, optionally following one balanced `<think>` block.
Whitespace is trimmed; punctuation, untagged prose, multiple answers, and malformed
tags are rejected. Save the untouched decoded output, including a second version
with special tokens, so all decisions can be reviewed.

A token-limit stop is recorded separately. A complete final answer at the limit
can still parse; missing/unparseable answers at the limit are flagged as possible
answer truncation. Invalid numerical scores invalidate the answer. Pilot evidence
may justify a documented parser revision before the final experiment.

In [ ]:
def parse_answer(text):
    """Strict, deterministic pilot parser; no label search inside reasoning."""
    response = text.strip()
    if not response:
        return None, "empty"
    # Exactly one balanced reasoning block may precede a final answer.
    if response.startswith("<think>"):
        if response.count("<think>") != 1 or response.count("</think>") != 1:
            return None, "incomplete_or_multiple_think_blocks"
        response = response.split("</think>", 1)[1].strip()
    if "<think>" in response or "</think>" in response:
        return None, "misplaced_think_tag"
    if "<answer>" in response or "</answer>" in response:
        match = re.fullmatch(r"<answer>\s*(benign|malignant)\s*</answer>", response, re.IGNORECASE)
        if match:
            return match.group(1).lower(), "answer_tag"
        return None, "malformed_or_ambiguous_answer_tag"
    match = re.fullmatch(r"(?:final answer:\s*)?(benign|malignant)", response, re.IGNORECASE)
    if match:
        return match.group(1).lower(), "direct_or_final_marker"
    return None, "no_unambiguous_final_answer"

def run_parser_checks():
    fixtures = [
        (" benign ", "benign"), ("MALIGNANT", "malignant"),
        ("Final answer: benign", "benign"),
        ("<think>Could be malignant.</think><answer>benign</answer>", "benign"),
        ("<think>Consider benign.</think>malignant", "malignant"),
        ("<answer>malignant</answer>", "malignant"),
        ("", None), ("benign or malignant", None), ("benign.", None),
        ("I think this is benign", None), ("<think>benign", None),
        ("<think>benign</think>", None), ("<answer>benign", None),
        ("<answer>benign</answer><answer>malignant</answer>", None),
        ("<answer>benign</answer> Final answer: malignant", None),
        ("<think><think>benign</think><answer>benign</answer>", None),
    ]
    for raw, expected in fixtures:
        actual, _ = parse_answer(raw)
        assert actual == expected, (raw, expected, actual)
    print(f"Passed {len(fixtures)} parser checks.")

run_parser_checks()



## 6. GPU checks, secret access, model loading, and per-image generation

Use native checkpoint processors/chat templates with a common task prompt; no
reference labels, case IDs, masks, BI-RADS, or metadata enter the model inputs.
Resolve model and processor to an exact shared Hub commit before loading. Record
actual device maps, parameter dtypes, image processing and quantized compute types.
TF32 is disabled for the FP32 comparison. CPU/disk offloading is disallowed.

Numerical diagnostics observe generation scores after built-in logits processors;
negative infinity may be legitimate masking. NaN, positive infinity, or entirely
masked scores invalidate a response. This probe adds some latency; timings are
engineering diagnostics, include preprocessing, and mark the first inference
following each load. No claim of peak optimized throughput is made.

In [ ]:
import importlib.metadata as im
import torch
from huggingface_hub import HfApi
from transformers import (AutoProcessor, Qwen2VLForConditionalGeneration,
                          Gemma3ForConditionalGeneration, BitsAndBytesConfig,
                          GenerationConfig, LogitsProcessor, LogitsProcessorList)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator, then restart and Run All.")
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
# Greedy generation is reproducible under a fixed setup, not guaranteed bit-identical across hardware.
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    # Public Qwen downloads may still work. Gemma access failure is recorded per job.
    print("HF_TOKEN is unavailable. Enable that Kaggle secret for the gated Gemma models.")

ENVIRONMENT = {
    "python": platform.python_version(), "platform": platform.platform(),
    "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
    "packages": {d.metadata["Name"]: d.version for d in im.distributions() if d.metadata["Name"]},
    "gpus": [{"index": i, "name": torch.cuda.get_device_name(i),
              "total_bytes": torch.cuda.get_device_properties(i).total_memory,
              "compute_capability": list(torch.cuda.get_device_capability(i))}
             for i in range(torch.cuda.device_count())],
    "tf32_enabled": False,
}
try:
    ENVIRONMENT["nvidia_smi"] = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=False).stdout.strip()
except OSError:
    ENVIRONMENT["nvidia_smi"] = "unavailable"
# Full package inventory is saved; only inference-relevant versions identify resume compatibility.
RUNTIME_IDENTITY = {"python": ENVIRONMENT["python"], "torch": ENVIRONMENT["torch"],
                    "cuda_runtime": ENVIRONMENT["cuda_runtime"], "gpus": ENVIRONMENT["gpus"],
                    "packages": {p: im.version(p) for p in [*PACKAGE_PINS, "torchvision", "numpy"]}}
write_json(OUTPUT_ROOT / "environment.json", ENVIRONMENT)
print("GPUs:", [(g["index"], g["name"]) for g in ENVIRONMENT["gpus"]])
print("No CPU/disk offloading or automatic precision fallback will be used.")

def sanitized_error(exc):
    message = str(exc)
    if HF_TOKEN:
        message = message.replace(HF_TOKEN, "[REDACTED]")
    message = re.sub(r"hf_[A-Za-z0-9]+", "[REDACTED]", message)
    return {"type": type(exc).__name__, "message": message[:4000]}

def cleanup_cuda():
    gc.collect()
    torch.cuda.empty_cache()

def sync_cuda():
    for i in range(torch.cuda.device_count()):
        torch.cuda.synchronize(i)

def reset_peaks():
    sync_cuda()
    for i in range(torch.cuda.device_count()):
        torch.cuda.reset_peak_memory_stats(i)

def memory_snapshot():
    return [{"gpu": i, "peak_allocated_bytes": torch.cuda.max_memory_allocated(i),
             "peak_reserved_bytes": torch.cuda.max_memory_reserved(i)}
            for i in range(torch.cuda.device_count())]

class NumericsProbe(LogitsProcessor):
    """Observe generation scores without modifying decoding or retaining tensors."""
    def __init__(self):
        self.steps = 0
        self.nonfinite_steps = 0
        self.nan_or_posinf_steps = 0
        self.all_masked_steps = 0

    def __call__(self, input_ids, scores):
        self.steps += 1
        self.nonfinite_steps += int((~torch.isfinite(scores)).any().item())
        self.nan_or_posinf_steps += int((torch.isnan(scores) | torch.isposinf(scores)).any().item())
        self.all_masked_steps += int(torch.isneginf(scores).all(dim=-1).any().item())
        return scores

def resolve_revision(key):
    path = OUTPUT_ROOT / "model_revisions.json"
    saved = json.loads(path.read_text()) if path.exists() else {}
    requested = MODEL_REVISIONS[key]
    if key in saved:
        if saved[key].get("model_id") != MODEL_IDS[key] or not re.fullmatch(r"[0-9a-f]{40}", saved[key].get("revision", "")):
            raise ValueError("Invalid saved checkpoint identity")
        if requested is not None and requested != saved[key]["revision"]:
            raise ValueError("Requested revision differs from frozen run; use a fresh OUTPUT_ROOT")
        return saved[key]["revision"]
    revision = HfApi(token=HF_TOKEN or False).model_info(MODEL_IDS[key], revision=requested).sha
    if not re.fullmatch(r"[0-9a-f]{40}", revision or ""):
        raise ValueError("Could not resolve exact model commit")
    saved[key] = {"model_id": MODEL_IDS[key], "revision": revision, "resolved_at": utc_now()}
    write_json(path, saved)
    return revision

def load_model(key, precision, revision):
    dtype = torch.float16 if precision == "fp16" else torch.float32
    # Use slow/native processor consistently to avoid silently changing image backends.
    cache = MODEL_CACHE_ROOT / key
    cache.mkdir(parents=True, exist_ok=True)
    (cache / ".bus_bra_owned_cache").touch()
    processor = AutoProcessor.from_pretrained(MODEL_IDS[key], revision=revision,
                                              token=HF_TOKEN or False, use_fast=False, cache_dir=cache)
    budget = {i: int(torch.cuda.mem_get_info(i)[0] * GPU_MEMORY_FRACTION)
              for i in range(torch.cuda.device_count())}
    kwargs = dict(revision=revision, token=HF_TOKEN or False, torch_dtype=dtype,
                  attn_implementation="sdpa", device_map="auto", max_memory={**budget, "cpu": 0},
                  low_cpu_mem_usage=True, cache_dir=cache)
    if precision == "int4_fp32":
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=False,
            bnb_4bit_compute_dtype=torch.float32,
        )
    cls = Qwen2VLForConditionalGeneration if key in {"qwen2_vl", "medvlm_r1"} else Gemma3ForConditionalGeneration
    model = cls.from_pretrained(MODEL_IDS[key], **kwargs).eval()
    device_map = {k: str(v) for k, v in model.hf_device_map.items()}
    if any(v in {"cpu", "disk"} for v in device_map.values()):
        raise RuntimeError("Model was offloaded to CPU/disk; this pilot requires GPU-only inference")
    details = {"device_map": device_map, "gpu_weight_budgets_bytes": budget,
               "parameter_elements_by_dtype": dict(Counter()),
               "quantization": model.config.to_dict().get("quantization_config"),
               "image_processor": processor.image_processor.to_dict(),
               "processor_class": type(processor).__name__,
               "chat_template": processor.chat_template,
               "special_tokens_map": processor.tokenizer.special_tokens_map,
               "generation_defaults": model.generation_config.to_dict()}
    dtypes = Counter()
    for parameter in model.parameters():
        dtypes[str(parameter.dtype)] += parameter.numel()
    details["parameter_elements_by_dtype"] = dict(dtypes)
    quant_layers = [{"name": name, "compute_dtype": str(layer.compute_dtype)}
                    for name, layer in model.named_modules() if hasattr(layer, "compute_dtype")]
    details["quantized_compute_layers"] = quant_layers
    if precision == "int4_fp32" and (not quant_layers or any(r["compute_dtype"] != "torch.float32" for r in quant_layers)):
        raise RuntimeError("4-bit compute configuration did not resolve to FP32")
    return model, processor, details, dtype

def infer_one(model, processor, dtype, image_path):
    # Deliberately accepts no manifest row, patient data, or reference answer.
    reset_peaks()
    started = time.perf_counter()
    with Image.open(image_path) as source:
        image = source.convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image"},
                 {"type": "text", "text": BUS_BRA_PROMPT}]}]
    rendered = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[rendered], images=[image], return_tensors="pt", padding=True)
    input_details = {k: list(v.shape) for k, v in inputs.items() if hasattr(v, "shape")}
    device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(device=device, dtype=dtype) if v.is_floating_point() else v.to(device)
              for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]
    # Fresh generation config prevents hidden checkpoint-specific sampling defaults.
    eos = model.generation_config.eos_token_id
    if eos is None: eos = processor.tokenizer.eos_token_id
    pad = processor.tokenizer.pad_token_id
    if pad is None: pad = eos[0] if isinstance(eos, list) else eos
    generation = GenerationConfig(max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                  num_beams=1, use_cache=True, eos_token_id=eos,
                                  pad_token_id=pad, bos_token_id=processor.tokenizer.bos_token_id)
    probe = NumericsProbe()
    with torch.inference_mode():
        output = model.generate(**inputs, generation_config=generation,
                                logits_processor=LogitsProcessorList([probe]))
    sync_cuda()
    latency_ms = (time.perf_counter() - started) * 1000
    tokens = output[0, input_len:].detach().cpu().tolist()
    raw = processor.tokenizer.decode(tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    raw_special = processor.tokenizer.decode(tokens, skip_special_tokens=False, clean_up_tokenization_spaces=False)
    eos_ids = eos if isinstance(eos, list) else [eos]
    stop = "eos" if tokens and tokens[-1] in eos_ids else "max_new_tokens" if len(tokens) >= MAX_NEW_TOKENS else "other"
    answer, reason = parse_answer(raw)
    bad_numerics = probe.nan_or_posinf_steps > 0 or probe.all_masked_steps > 0
    if bad_numerics:
        answer, reason = None, "invalid_generation_scores"
    counts = Counter(tokens)
    return {"raw_output": raw, "raw_output_with_special_tokens": raw_special,
            "parsed_answer": answer, "parse_status": "parsed" if answer is not None else "unparsed",
            "parse_reason": reason, "generated_tokens": len(tokens), "generated_token_ids": tokens,
            "stop_reason": stop, "hit_token_limit": stop == "max_new_tokens",
            "possible_answer_truncation": stop == "max_new_tokens" and answer is None,
            "latency_ms": latency_ms, "peak_memory": memory_snapshot(),
            "input_shapes": input_details, "rendered_prompt_sha256": sha_bytes(rendered.encode()),
            "score_diagnostics": vars(probe), "invalid_numerics": bad_numerics,
            "repetition_candidate": len(tokens) >= 8 and max(counts.values(), default=0) / len(tokens) >= 0.5,
            "generation_config": generation.to_dict()}



## 7. Resumable runner and result export

Completed rows are saved immediately and skipped only if the sample, model
revision, source/manifest, full configuration, code, and runtime identities match.
Malformed complete records or duplicate IDs stop reuse. A interrupted final JSONL
fragment is retained separately and repaired. Infrastructure failures go to event
logs; they do not become fabricated completed answers.

OOM stops the affected job. Three consecutive per-image errors stop that job.
A device-side CUDA assert/illegal memory access stops the session's remaining
jobs: save outputs, restart the runtime, and resume. Normal load/access errors
are retained while the remaining planned jobs continue.

A `complete` status means 20 responses were saved, **not** that a model passed.
The bundle is refreshed after each job so partial results are also reviewable.
If the whole Kaggle process is killed, only previously persisted output survives;
save versions/outputs between shorter sessions when necessary.

In [ ]:
def validate_completed(rows, identity, expected_ids):
    completed = {}
    for row in rows:
        if row.get("sample_id") not in expected_ids:
            raise ValueError("Saved response not in pilot manifest")
        if row["sample_id"] in completed:
            raise ValueError("Duplicate completed sample in saved responses")
        for key, value in identity.items():
            if row.get(key) != value:
                raise ValueError(f"Saved response has incompatible {key}")
        if row.get("parse_status") not in {"parsed", "unparsed"} or not isinstance(row.get("raw_output"), str):
            raise ValueError("Malformed saved response")
        if row["parse_status"] == "parsed" and row.get("parsed_answer") not in {"benign", "malignant"}:
            raise ValueError("Invalid saved parsed answer")
        if row["parse_status"] == "unparsed" and row.get("parsed_answer") is not None:
            raise ValueError("Unparsed response must have a null answer")
        completed[row["sample_id"]] = row
    return completed

def run_job(key, precision):
    model = processor = None
    run_dir = None
    revision = None
    status = "failed_before_run"
    metadata = None
    completed = {}
    started = time.perf_counter()
    fatal_cuda = False
    try:
        cleanup_cuda()
        revision = resolve_revision(key)
        config = {
            "purpose": "bus_bra_engineering_pilot", "model_key": key, "model_id": MODEL_IDS[key],
            "model_revision": revision, "processor_revision": revision,
            "precision": precision, "quantization": ("nf4_no_double_fp32_compute" if precision == "int4_fp32" else None),
            "manifest_sha256": MANIFEST_SHA, "source_fingerprint": AUDIT["source_fingerprint"],
            "prompt": BUS_BRA_PROMPT, "parser_version": PARSER_VERSION,
            "max_new_tokens": MAX_NEW_TOKENS, "do_sample": False, "batch_size": 1,
            "seed": SEED, "attention": "sdpa", "processor_use_fast": False,
            "gpu_memory_fraction": GPU_MEMORY_FRACTION, "cpu_offload": False,
            "tf32": False, "runtime": RUNTIME_IDENTITY,
            "code_version": PILOT_CODE_VERSION, "implementation_sha256": IMPLEMENTATION_SHA256,
            "code_git_commit": CODE_GIT_COMMIT,
        }
        config_sha = object_hash(config)
        run_id = f"{key}__{precision}__{config_sha[:12]}"
        run_dir = OUTPUT_ROOT / "runs" / run_id
        run_dir.mkdir(parents=True, exist_ok=True)
        config_path = run_dir / "resolved_config.json"
        if config_path.exists() and json.loads(config_path.read_text()) != config:
            raise ValueError("Existing run configuration conflicts")
        write_json(config_path, config)
        identity = {"run_id": run_id, "dataset": "bus_bra", "model_key": key,
                    "model_id": MODEL_IDS[key], "model_revision": revision,
                    "manifest_sha256": MANIFEST_SHA, "config_sha256": config_sha}
        expected_ids = {r["sample_id"] for r in PILOT}
        completed = validate_completed(read_jsonl(run_dir / "generations.jsonl", repair_tail=True),
                                       identity, expected_ids)
        metadata_path = run_dir / "metadata.json"
        metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {
            **identity, "precision": precision, "expected": 20, "attempts": []}
        metadata["attempts"].append({"started_at": utc_now(), "already_completed": len(completed)})
        attempt = metadata["attempts"][-1]
        if len(completed) == 20:
            status = "complete"
            print(f"{key}/{precision}: all 20 responses already saved; skipping model load")
        else:
            metadata.update(status="loading", completed=len(completed))
            write_json(metadata_path, metadata)
            reset_peaks()
            load_started = time.perf_counter()
            model, processor, details, dtype = load_model(key, precision, revision)
            sync_cuda()
            attempt["load_wall_ms_including_download"] = (time.perf_counter() - load_started) * 1000
            attempt["load_peak_memory"] = memory_snapshot()
            metadata["loaded_model"] = details
            metadata["score_probe_note"] = "Post-logits-processor scores; negative infinity can be legitimate masking. NaN/positive infinity/all-masked scores are invalid."
            write_json(metadata_path, metadata)
            new_count = 0
            consecutive_errors = 0
            for row in PILOT:
                if row["sample_id"] in completed:
                    continue
                try:
                    payload = infer_one(model, processor, dtype, SOURCE_ROOT / row["image_path"])
                    record = {**identity, "sample_id": row["sample_id"], "precision": precision,
                              "first_inference_after_load": new_count == 0,
                              "completed_at": utc_now(), **payload}
                    append_jsonl(run_dir / "generations.jsonl", record)
                    completed[row["sample_id"]] = record
                    new_count += 1
                    consecutive_errors = 0
                    print(f"  {key}/{precision} {len(completed)}/20: {record['parse_status']}; {record['generated_tokens']} tokens; {record['stop_reason']}")
                except Exception as exc:
                    error = sanitized_error(exc)
                    append_jsonl(run_dir / "events.jsonl", {"at": utc_now(), "stage": "inference",
                                 "sample_id": row["sample_id"], **error})
                    consecutive_errors += 1
                    message = str(exc).lower()
                    fatal_cuda = "device-side assert" in message or "illegal memory access" in message
                    print(f"  {key}/{precision}: {error['type']} on {row['sample_id']} (saved to events)")
                    if fatal_cuda or "out of memory" in message or consecutive_errors >= 3:
                        break
                metadata.update(status="running", completed=len(completed))
                write_json(metadata_path, metadata)
            status = "complete" if len(completed) == 20 else "incomplete"
        attempt["finished_at"] = utc_now()
        attempt["wall_ms"] = (time.perf_counter() - started) * 1000
        metadata.update(status=status, completed=len(completed))
        write_json(run_dir / "metadata.json", metadata)
        missing = sorted({r["sample_id"] for r in PILOT} - completed.keys())
        write_json(run_dir / "validation.json", {"expected": 20, "completed": len(completed),
                                                "missing_sample_ids": missing, "coverage_complete": not missing})
        return {"model_key": key, "precision": precision, "status": status,
                "run_id": run_id, "completed": len(completed), "fatal_cuda": fatal_cuda}
    except Exception as exc:
        error = sanitized_error(exc)
        fatal_cuda = any(s in str(exc).lower() for s in ("device-side assert", "illegal memory access"))
        event = {"at": utc_now(), "model_key": key, "precision": precision,
                 "stage": "job", "revision": revision, **error}
        append_jsonl(OUTPUT_ROOT / "job_failures.jsonl", event)
        if run_dir is not None:
            append_jsonl(run_dir / "events.jsonl", event)
            if metadata is not None:
                metadata.update(status="failed", completed=len(completed), last_error=error)
                if metadata.get("attempts"):
                    metadata["attempts"][-1].update(finished_at=utc_now(),
                                                   wall_ms=(time.perf_counter() - started) * 1000)
                    if not fatal_cuda:
                        try:
                            metadata["attempts"][-1]["failure_peak_memory"] = memory_snapshot()
                        except Exception:
                            pass
                write_json(run_dir / "metadata.json", metadata)
        print(f"{key}/{precision}: {error['type']}: {error['message'][:240]}")
        return {"model_key": key, "precision": precision, "status": "failed",
                "run_id": run_dir.name if run_dir else None, "completed": len(completed),
                "fatal_cuda": fatal_cuda, "error_type": error["type"], "error": error["message"]}
    finally:
        model = processor = None
        if not fatal_cuda:
            cleanup_cuda()

def export_review_bundle():
    # Read all retained compatible runs. If multiple configurations exist, retain run IDs;
    # never silently select one as the final scientific result.
    summaries, per_case, comparisons = [], [], []
    group_runs = defaultdict(list)
    for run_dir in sorted((OUTPUT_ROOT / "runs").glob("*")):
        config_path = run_dir / "resolved_config.json"
        if not config_path.exists(): continue
        config = json.loads(config_path.read_text())
        rows = read_jsonl(run_dir / "generations.jsonl", repair_tail=True)
        meta_path = run_dir / "metadata.json"
        meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
        if config["manifest_sha256"] != MANIFEST_SHA:
            continue
        identity = {"run_id": run_dir.name, "dataset": "bus_bra", "model_key": config["model_key"],
                    "model_id": config["model_id"], "model_revision": config["model_revision"],
                    "manifest_sha256": MANIFEST_SHA, "config_sha256": object_hash(config)}
        completed = validate_completed(rows, identity, {r["sample_id"] for r in PILOT})
        rows = list(completed.values())
        invalid = sum(r["parse_status"] == "unparsed" for r in rows)
        summary = {"run_id": run_dir.name, "model_key": config["model_key"], "precision": config["precision"],
                   "status": meta.get("status", "unknown"), "expected": 20, "completed": len(rows),
                   "invalid": invalid, "invalid_rate_completed": invalid / len(rows) if rows else None,
                   "token_limit_stops": sum(r["hit_token_limit"] for r in rows),
                   "possible_answer_truncations": sum(r["possible_answer_truncation"] for r in rows),
                   "numerically_invalid": sum(r["invalid_numerics"] for r in rows),
                   "repetition_candidates": sum(r["repetition_candidate"] for r in rows),
                   "mean_generated_tokens": statistics.mean(r["generated_tokens"] for r in rows) if rows else None,
                   "median_latency_ms": statistics.median(r["latency_ms"] for r in rows) if rows else None,
                   "last_load_wall_ms_including_download": next((a["load_wall_ms_including_download"] for a in reversed(meta.get("attempts", [])) if "load_wall_ms_including_download" in a), None),
                   "max_single_gpu_allocated_GiB": max((m["peak_allocated_bytes"] / 2**30 for r in rows for m in r["peak_memory"]), default=None),
                   "needs_review": len(rows) != 20 or invalid > 0 or any(r["hit_token_limit"] or r["invalid_numerics"] for r in rows)}
        summaries.append(summary)
        for r in rows:
            per_case.append({k: r[k] for k in ("run_id", "model_key", "precision", "sample_id", "raw_output",
                                              "parsed_answer", "parse_status", "parse_reason", "generated_tokens",
                                              "stop_reason", "hit_token_limit", "possible_answer_truncation", "latency_ms")})
        group_runs[(config["model_key"], config["precision"])].append((config, completed, run_dir.name))
    for model in ("gemma3", "medgemma"):
        for a, rows_a, id_a in group_runs[(model, "fp32")]:
            for b, rows_b, id_b in group_runs[(model, "int4_fp32")]:
                # Only compare jobs with matching model, data, prompt, code, and runtime.
                remove = {"precision", "quantization"}
                if {k: v for k, v in a.items() if k not in remove} != {k: v for k, v in b.items() if k not in remove}:
                    continue
                for sample_id in sorted(rows_a.keys() & rows_b.keys()):
                    left, right = rows_a[sample_id], rows_b[sample_id]
                    valid = left["parse_status"] == right["parse_status"] == "parsed"
                    comparisons.append({"model_key": model, "sample_id": sample_id, "fp32_run": id_a,
                                        "int4_run": id_b, "fp32_answer": left["parsed_answer"],
                                        "int4_answer": right["parsed_answer"], "both_parsed": valid,
                                        "answer_disagreement": left["parsed_answer"] != right["parsed_answer"] if valid else None,
                                        "parse_status_disagreement": left["parse_status"] != right["parse_status"],
                                        "raw_text_disagreement": left["raw_output"] != right["raw_output"]})
    write_csv(OUTPUT_ROOT / "pilot_summary.csv", summaries)
    write_csv(OUTPUT_ROOT / "per_case_review.csv", per_case)
    write_csv(OUTPUT_ROOT / "precision_comparison.csv", comparisons,
              ["model_key", "sample_id", "fp32_run", "int4_run", "fp32_answer", "int4_answer",
               "both_parsed", "answer_disagreement", "parse_status_disagreement", "raw_text_disagreement"])
    write_json(OUTPUT_ROOT / "review_notes.json", {
        "purpose": "smoke/pilot feasibility only; not final evaluation",
        "counts": {"pilot_images": 20, "planned_jobs_this_session": len(JOBS), "retained_runs": len(summaries)},
        "interpretation": ["Complete means 20 raw responses saved, not that answers are valid or models passed.",
                           "No automatic precision selection or token-cap changes were performed.",
                           "Token-limit stopping is separate from provable missing/incomplete answers.",
                           "Inspect failures, raw responses, contact sheet and audits before any final experiment.",
                           "Timing includes preprocessing and score diagnostics; first inference includes cold-start overhead.",
                           "Precision comparison is preliminary and does not establish accuracy equivalence."],
    })
    zip_path = OUTPUT_ROOT.parent / (OUTPUT_ROOT.name + "_results.zip")
    temp = zip_path.with_suffix(".tmp")
    with zipfile.ZipFile(temp, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT_ROOT.rglob("*")):
            if path.is_file() and path.suffix != ".tmp":
                archive.write(path, Path(OUTPUT_ROOT.name) / path.relative_to(OUTPUT_ROOT))
    os.replace(temp, zip_path)
    return summaries, zip_path



## 8. Run the selected smoke-test jobs

This is the only cell that downloads model weights and runs GPU inference.
All selected jobs read the same pilot manifest. It can be rerun to retry failed
cases with identical settings. Different runtime/settings create different run IDs.
The Hugging Face cache is not included in results; owned model caches are removed
after their final scheduled job, so retries may need to download weights again.

In [ ]:
SESSION_RESULTS = []
write_json(OUTPUT_ROOT / "requested_jobs.json", [{"model_key": k, "precision": p} for k, p in JOBS])
try:
    for index, (key, precision) in enumerate(JOBS):
        print(f"\n=== {index + 1}/{len(JOBS)}: {key} / {precision} ===")
        outcome = run_job(key, precision)
        SESSION_RESULTS.append(outcome)
        write_json(OUTPUT_ROOT / "session_results.json", SESSION_RESULTS)
        export_review_bundle()  # Refresh downloadable evidence after every job.
        if CLEAR_MODEL_CACHE_AFTER_LAST_JOB and key not in {k for k, _ in JOBS[index + 1:]}:
            cache = MODEL_CACHE_ROOT / key
            if cache.is_dir() and not cache.is_symlink() and (cache / ".bus_bra_owned_cache").is_file():
                shutil.rmtree(cache)  # Only weights downloaded into this notebook's owned cache.
        if outcome.get("fatal_cuda"):
            print("CUDA context error: stop here, save outputs, restart session and resume from saved outputs.")
            break
finally:
    PILOT_SUMMARIES, RESULTS_ZIP = export_review_bundle()
    print("\nReview bundle:", RESULTS_ZIP)



## 9. Review and download results

Download **`bus_bra_smoke_results.zip`**. It contains:
- `pilot_summary.csv`, `per_case_review.csv`, and `precision_comparison.csv`;
- `pilot_manifest.csv`, `reserved_pilot_cases.csv`, source hashes and contact sheet;
- `runs/<run_id>/generations.jsonl`, resolved settings, metadata, events, coverage;
- `environment.json`, frozen `model_revisions.json`, session outcomes and failures;
- `audit/` records including exclusions and duplicate candidates.

Bring the ZIP back for review. We will check coverage, GPU feasibility, answer
format/truncation and precision disagreements before deciding the next step.
No final-test accuracy or statistical comparison is produced by this notebook.

To resume: save the notebook with outputs; add that notebook's output as an input
in a new session (or privately upload the extracted ZIP folder), set `RESUME_FROM`
to the attached `bus_bra_smoke` directory, and Run All. Set `JOBS` to pending jobs.
A new GPU/runtime creates separate run IDs rather than combining incompatible runs.
Keep outputs private until identifiers and dataset terms have been reviewed.

In [ ]:
import pandas as pd
from IPython.display import FileLink, display

PILOT_SUMMARIES, RESULTS_ZIP = export_review_bundle()
display(pd.DataFrame(SESSION_RESULTS))
display(pd.DataFrame(PILOT_SUMMARIES))
# FileLink works from the /kaggle/working notebook directory; Output files is a fallback.
try:
    display(FileLink(str(RESULTS_ZIP.relative_to(Path.cwd()))))
except ValueError:
    print("Download from Kaggle Output files:", RESULTS_ZIP)
print("Save this notebook version with outputs and download the ZIP. Do not start final inference yet.")



## Implementation references

- [Qwen2-VL model and usage](https://huggingface.co/Qwen/Qwen2-VL-2B-Instruct)
- [MedVLM-R1 model and reasoning output](https://huggingface.co/JZPeterPan/MedVLM-R1)
- [MedGemma model and access](https://huggingface.co/google/medgemma-4b-it)
- [Transformers 4.57.1 Gemma 3](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/gemma3)
- [Transformers 4.57.1 bitsandbytes quantization](https://huggingface.co/docs/transformers/v4.57.1/en/quantization/bitsandbytes)
- [BUS-BRA original dataset](https://zenodo.org/records/8231412)

BUS-BRA attribution: Gómez-Flores, Gregorio-Calas, and Pereira, *BUS-BRA: A Breast
Ultrasound Dataset for Assessing Computer-aided Diagnosis Systems*, Medical Physics
51, 3110–3123 (2024), DOI: 10.1002/mp.16812. Retain the source dataset license when
redistributing its contents.